# 01 — Exploratory Data Analysis

This notebook walks through initial data exploration: shape, dtypes, missing values, distributions, correlations, and target analysis.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

DATA_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')

## 1. Load Data

In [ ]:
from sklearn.datasets import load_breast_cancer

raw = load_breast_cancer(as_frame=True)
df = raw.frame
df['target'] = raw.target

print(f'Shape: {df.shape}')
df.head()

## 2. Data Overview

In [ ]:
print('=== dtypes ===')
print(df.dtypes.value_counts())
print('\n=== Missing values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values')
print('\n=== Descriptive stats ===')
df.describe()

## 3. Target Distribution

In [ ]:
target_counts = df['target'].value_counts().reset_index()
target_counts.columns = ['class', 'count']
fig = px.bar(target_counts, x='class', y='count', title='Target Class Distribution', color='class', text='count')
fig.update_traces(textposition='outside')
fig.show()

## 4. Feature Distributions

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'target']
sample_cols = numeric_cols[:10]
fig = make_subplots(rows=2, cols=5, subplot_titles=sample_cols)
for i, col in enumerate(sample_cols):
    row, col_idx = divmod(i, 5)
    fig.add_trace(go.Histogram(x=df[col], name=col, showlegend=False, nbinsx=30), row=row+1, col=col_idx+1)
fig.update_layout(height=500, title_text='Feature Distributions (first 10)')
fig.show()

## 5. Correlation Matrix

In [ ]:
corr = df[numeric_cols[:15]].corr()
fig = px.imshow(corr, title='Feature Correlation Matrix (first 15 features)', color_continuous_scale='RdBu_r', zmin=-1, zmax=1, aspect='auto')
fig.show()

## 6. Feature vs Target

In [ ]:
top_cols = numeric_cols[:6]
fig = px.box(df.melt(id_vars='target', value_vars=top_cols), x='variable', y='value', color='target', title='Feature Distributions by Target Class')
fig.update_layout(xaxis_tickangle=-30)
fig.show()

## 7. Save Processed Snapshot

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df.to_parquet(PROCESSED_DIR / 'eda_snapshot.parquet', index=False)
print(f'Saved {len(df)} rows to {PROCESSED_DIR / "eda_snapshot.parquet"}')